# M16 · 03 — Task corpus and Qwen teacher outputs (A0)

Decision: `docs/decisions/0025-own-llm-m16.md` §6. Produced by `python -m llm.corpus process --wait` (TRAINING.md §5c).

1. Processing progress and cost per minute of audio (Whisper + chapters + show notes + relevance).
2. Recorded teacher calls per task, and how many are usable SFT examples (`llm.task_data`).
3. **Do the app's prompts + teacher outputs fit our 4,096-token context with our tokenizer?** (stage A pretrains at 2,048; task SFT runs at 4,096).
4. Split sanity: test channels and the 0024 test episodes never in train.

Re-run as processing advances; the final run is recorded in the conclusions.

In [1]:
import json, statistics
from collections import Counter, defaultdict
from pathlib import Path
from llm.corpus import CORPUS, episodes

eps = episodes()
metas = [json.loads(p.read_text()) for p in sorted((CORPUS / 'processed').glob('*/meta.json'))]
audio_min = sum(m['duration'] for m in metas) / 60
print(f"processed {len(metas)} / {len(eps)} episodes, {audio_min / 60:.1f} h of audio")
for step in ('transcribe_s', 'chapters_s', 'shownotes_s', 'relevance_s'):
    tot = sum(m['steps'].get(step, 0) for m in metas if m['steps'].get('transcribe_s', 0) > 0)
    mins = sum(m['duration'] for m in metas if m['steps'].get('transcribe_s', 0) > 0) / 60
    print(f"  {step:13s} {tot / mins:5.2f} s per audio minute")
print(f"  total        {sum(m['seconds'] for m in metas) / audio_min:5.2f} s per audio minute")
remaining_min = sum((e.get('duration') or 2400) for e in eps if e['id'] not in {m['id'] for m in metas}) / 60
print(f"remaining ~{remaining_min / 60:.0f} h of audio -> ~{remaining_min * sum(m['seconds'] for m in metas) / audio_min / 3600:.1f} h of GPU")
print('by split:', Counter(m['split'] for m in metas))
print('by channel:', dict(Counter(m['channel'] for m in metas)))

processed 59 / 348 episodes, 59.2 h of audio
  transcribe_s   1.29 s per audio minute
  chapters_s     0.15 s per audio minute
  shownotes_s    0.79 s per audio minute
  relevance_s    1.26 s per audio minute
  total         3.47 s per audio minute
remaining ~340 h of audio -> ~19.7 h of GPU
by split: Counter({'train': 56, 'val': 2, 'test': 1})
by channel: {'the-wild-project': 22, 'la-pija-y-la-quinqui': 26, 'estirando-el-chicle': 10, 'sastre-y-maldonado': 1}


In [2]:
from llm.task_data import build, subtask, usable
from llm import HOME
stats = build(HOME / 'data' / 'sft' / 'tasks')
for k, v in stats.items():
    print(f"{k:28s} {v:7,d}")

episodes_test                      1
episodes_train                    56
episodes_val                       2
test_relevance                    12
test_shownotes_map                 4
test_shownotes_reduce              1
test_title                         3
train_relevance                1,428
train_shownotes_map              601
train_shownotes_reduce            56
train_title                      363
val_relevance                     46
val_shownotes_map                 18
val_shownotes_reduce               2
val_title                         11


In [3]:
from llm.prepare_data import tokenizer_path
from llm.sft_data import encode_example, read_jsonl
from llm.tokenizer import load_tokenizer
tok = load_tokenizer(tokenizer_path())

lens = defaultdict(list)
targets = defaultdict(list)
for ex in read_jsonl(HOME / 'data' / 'sft' / 'tasks' / 'tasks_train.jsonl'):
    ids, mask = encode_example(tok, ex['messages'])
    lens[ex['task']].append(len(ids))
    targets[ex['task']].append(sum(mask))
print(f"{'task':18s} {'n':>6s} {'median':>7s} {'p95':>6s} {'max':>6s} {'>2048':>6s} {'>4096':>6s} {'target med':>10s}")
for t, L in sorted(lens.items()):
    s = sorted(L)
    print(f"{t:18s} {len(s):6d} {statistics.median(s):7.0f} {s[int(0.95 * (len(s) - 1))]:6d} {s[-1]:6d} "
          f"{sum(x > 2048 for x in s):6d} {sum(x > 4096 for x in s):6d} {statistics.median(targets[t]):10.0f}")

task                    n  median    p95    max  >2048  >4096 target med
relevance            1428    1794   2101   2279    144      0         68
shownotes_map         601    2096   2609   2824    304      0        205
shownotes_reduce       56    1496   2975   3011     23      0        413
title                 363    1352   1520   1672      0      0         11


In [4]:
from llm.corpus import TEST_CHANNELS, SYM_SPLIT
split = json.loads((CORPUS / 'split.json').read_text())
sym_test = set(json.loads(SYM_SPLIT.read_text())['split_0024']['test'])
bad = [e['id'] for e in eps if (e['channel'] in TEST_CHANNELS or e['id'] in sym_test) and split.get(e['id']) != 'test']
print('test-channel / 0024-test episodes outside test:', bad)
print('split sizes:', Counter(split.values()))

test-channel / 0024-test episodes outside test: []
split sizes: Counter({'train': 283, 'test': 49, 'val': 16})


In [5]:
# One example per task, truncated, to eyeball what the model will learn.
seen = set()
for ex in read_jsonl(HOME / 'data' / 'sft' / 'tasks' / 'tasks_train.jsonl'):
    if ex['task'] in seen:
        continue
    seen.add(ex['task'])
    user = ex['messages'][-2]['content']
    print(f"=== {ex['task']} ({ex['channel']}) ===\nUSER: {user[:300]}...\nASSISTANT: {ex['messages'][-1]['content'][:400]}\n")

=== title (the-wild-project) ===
USER: Esta es la transcripción de un capítulo:

"""
¿Qué tal mi gente? ¿Cómo estamos? Esta es una breve introducción a las tres entrevistas con los candidatos a la presidencia de España. Hago la misma introducción para todos, simplemente para explicaros el formato, por si no lo sabíais. La duración está p...
ASSISTANT: Introducción a las entrevistas con los candidatos a la presidencia

=== shownotes_map (the-wild-project) ===
USER: Esta es una parte de la transcripción del episodio (capítulo "Introducción a las entrevistas con los candidatos a la presidencia"):

"""
¿Qué tal mi gente? ¿Cómo estamos? Esta es una breve introducción a las tres entrevistas con los candidatos a la presidencia de España. Hago la misma introducción p...
ASSISTANT: *   La duración de las entrevistas está pactada entre 60 y 90 minutos, aunque se aceptaron ofertas de candidatos para extenderlas.
*   La entrevista más corta será la de Feijóo debido a compromisos previos que limita

In [6]:
# Teacher quality check (relevance): how often does Qwen answer with a near-monotonic ramp
# (e.g. 15, 10, 10, ..., 80, 85) instead of scoring each fragment? Our model would copy that pattern.
import numpy as np
mono, n, spread = 0, 0, []
for ex in read_jsonl(HOME / 'data' / 'sft' / 'tasks' / 'tasks_train.jsonl'):
    if ex['task'] != 'relevance':
        continue
    s = np.array(json.loads(ex['messages'][-1]['content'])['scores'])
    d = np.diff(s)
    n += 1
    mono += (d >= 0).mean() >= 0.9 or (d <= 0).mean() >= 0.9
    spread.append(s.std())
print(f"relevance batches: {n}; >=90 % monotonic: {mono} ({mono / n:.0%}); median std of the 20 scores: {np.median(spread):.1f}")
from scipy.stats import spearmanr
rho = []
for ex in read_jsonl(HOME / 'data' / 'sft' / 'tasks' / 'tasks_train.jsonl'):
    if ex['task'] == 'relevance':
        s = json.loads(ex['messages'][-1]['content'])['scores']
        rho.append(spearmanr(range(len(s)), s).statistic)
rho = np.nan_to_num(np.array(rho))
print(f"Spearman(score, position in batch): median {np.median(rho):.2f}; share > 0.8: {(rho > 0.8).mean():.0%}")

relevance batches: 1428; >=90 % monotonic: 22 (2%); median std of the 20 scores: 11.2


/tmp/ipykernel_27962/406449850.py:11: RuntimeWarning: Mean of empty slice
  mono += (d >= 0).mean() >= 0.9 or (d <= 0).mean() >= 0.9
/home/pablo/projects/new_youtube_suite/backend/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


/tmp/ipykernel_27962/406449850.py:19: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho.append(spearmanr(range(len(s)), s).statistic)


Spearman(score, position in batch): median 0.28; share > 0.8: 9%


## Conclusions (interim, 2026-11-14, 59 of 348 episodes processed)

- **Cost:** 3.47 s per audio minute in total. Whisper 1.29, relevance 1.26, show notes 0.79,
  chapters 0.15. The remaining ~340 h of audio need about 20 h of GPU.
- **Teacher calls are all usable** so far (0 dropped by `llm.task_data.usable`).
  - From 56 train episodes: 1,428 relevance batches, 601 show-notes map calls, 56 reduce calls and
    363 titles.
  - At about 348 episodes this extrapolates to roughly 9k relevance, 3.7k map, 350 reduce and 2.3k
    title examples.
  - **Reduce is the scarce task** (one per episode). Expect to lean on general-SFT JSON data and DPO
    for it (0025 §9).
- **Context:** every example fits 4,096 tokens with our tokenizer (max 3,011).
  - 30–50 % of the map and relevance examples are longer than 2,048. Task SFT therefore runs at
    `seq_len` 4,096, which also acts as stage A's context extension. Stage A pretrains at 2,048.
- **Teacher quality (relevance):**
  - 2 % of batches are near-monotonic ramps, 9 % correlate strongly with position (Spearman > 0.8),
    and the median position Spearman is 0.28. This is a mild position bias in Qwen's zero-shot
    scores.
  - Our model will learn it too. **Option (not applied yet):** drop batches with position
    Spearman > 0.8 from task SFT. To decide at A3 from the relevance gate (0025 §10).
- **Split:** the test channels and the 0024 test episodes are all in `test` (checked). Frozen sizes:
  283 train / 16 val / 49 test episodes.